# NEX-GDDP-CMIP6 Dataset Ingestion

This notebook builds GDAL VRTs over the yearly NEX-GDDP-CMIP6 NetCDF files on S3 and registers them as GeoEngine datasets. Each year has one raster band per day; the VRT concatenates them so band *i* == day *i* of the period, readable via NetCDF-CF metadata without kerchunk or a GDAL >= 3.11.

## Prerequisites

- Python 3.10+
- Required packages: `s3fs`, `fsspec`, `geoengine`, `rasterio` (only for the optional read-back check)
- Access to the NEX-GDDP-CMIP6 S3 bucket (public, no credentials required)
- GeoEngine instance with dataset creation permissions


In [1]:
import json
from datetime import datetime
from pathlib import Path

import fsspec
import s3fs

import geoengine as ge
import geoengine_api_client as geoc

## Configuration

Update these paths to match your environment.

In [2]:
# Unused unless the volume type requires an upload flow (no filesystem path);
# keep as the local staging dir for that case.
REFERENCE_PATH = Path("/data/nexgddp-cmip6-refs")

# NEX-GDDP-CMIP6 S3 bucket (public)
S3_BUCKET = "nex-gddp-cmip6"
S3_PREFIX = "NEX-GDDP-CMIP6"

# GeoEngine connection
GEOENGINE_URL = "http://localhost:3030/api"
GEOENGINE_USER = "admin@localhost"
GEOENGINE_PASSWORD = "adminadmin"

# Volume name in GeoEngine
VOLUME_NAME = "ge_geo_data"

# Full model registry: the authoritative id/variant/grid/scenario list. The dev registry
# (`nexgddp_models.toml`) is trimmed to what is ingested, so ingest reads this one and the
# trimmed file is regenerated from it afterwards.
REGISTRY_PATH = Path("../backend/conf/nexgddp_models.full.toml")

try:
    import tomllib
except ModuleNotFoundError:  # Python < 3.11
    import tomli as tomllib

with open(REGISTRY_PATH, "rb") as fh:
    REGISTRY = tomllib.load(fh)["model"]

MODELS = [
    {"id": m["id"], "variant": m["variant"], "grid": m["grid"]} for m in REGISTRY
]

SCENARIOS = sorted({s for m in REGISTRY for s in m["scenarios"]})

# Variables
VARIABLES = ["tasmax", "tasmin", "pr"]

# Variable metadata
VARIABLE_CONFIGS = {
    "tasmax": {
        "name": "Maximum Temperature",
        "measurement": "temperature",
        "unit": "K",
        "min": 200,
        "max": 340,
        "color_map": "coolwarm",
    },
    "tasmin": {
        "name": "Minimum Temperature",
        "measurement": "temperature",
        "unit": "K",
        "min": 200,
        "max": 340,
        "color_map": "coolwarm",
    },
    "pr": {
        "name": "Precipitation",
        "measurement": "precipitation",
        "unit": "kg m-2 s-1",
        "min": 0,
        "max": 0.01,
        "color_map": "Blues",
    },
}
# ponytail: trial mode keeps AWS traffic low. Set DEV_ONLY = False for the full matrix
# (all 5 models x 3 scenarios x 3 variables).
DEV_ONLY = True
if DEV_ONLY:
    MODELS = MODELS[:1]
    SCENARIOS = SCENARIOS[:2]


## Initialize GeoEngine Connection

In [3]:
ge.initialize(
    GEOENGINE_URL,
    credentials=(GEOENGINE_USER, GEOENGINE_PASSWORD),
)

volume = ge.volume_by_name(VOLUME_NAME)
print(f"Using volume: {VOLUME_NAME} (path={volume.path!r})")

# Reference files must live where GeoEngine can reach them. If the volume has a
# filesystem path, write refs under it; otherwise an upload step is required.
DATA_DIR = Path(volume.path) / "nexgddp-cmip6" if volume.path else None
if DATA_DIR is None:
    print("Warning: volume has no filesystem path - reference files must be uploaded")

permission_tuples = [
    (ge.permissions.ANONYMOUS_USER_ROLE_ID, ge.Permission.READ),
    (ge.permissions.REGISTERED_USER_ROLE_ID, ge.Permission.READ),
]

Using volume: ge_geo_data (path='/mnt/peter/geo_data/')


## Build VRTs

For each model/scenario/variable combination, build a GDAL VRT that concatenates the daily bands of the yearly NetCDF files, streamed from S3 via `/vsicurl`.

In [4]:
def list_netcdf_files(model_id, scenario, variable, variant, grid):
    """List all yearly NetCDF files for a model/scenario/variable combination."""
    fs = s3fs.S3FileSystem(anon=True)
    
    # Determine year range based on scenario
    if scenario == "historical":
        years = range(1950, 2015)
    else:
        years = range(2015, 2101)
    
    files = []
    for year in years:
        # NEX-GDDP-CMIP6 published corrected `{year}_v2.0.nc` copies for early
        # historical years; prefer them over the uncorrected base file.
        base = (f"{S3_BUCKET}/{S3_PREFIX}/{model_id}/{scenario}/{variant}/{variable}/"
                f"{variable}_day_{model_id}_{scenario}_{variant}_{grid}_")
        path = next(
            (base + cand for cand in (f"{year}_v2.0.nc", f"{year}.nc")
             if fs.exists(base + cand)),
            None,
        )
        if path is not None:
            files.append(path)
        else:
            print(f"Warning: File not found: {base}{year}.nc")
    
    return sorted(files)

In [5]:

def create_vrt(files, output_path):
    """Build a VRT concatenating the daily bands of yearly NetCDF files.

    Each yearly file contributes 365/366 bands (one per day). Concatenated in
    order, band i == day i since the first year, which is exactly what the
    NetCDF-CF metadata (band = 1 + band_offset + steps_between) expects.
    Sources use the canonical VRT ComplexSource form so GDAL actually sources
    the bands (a bare <SourceFilename>/<SourceBand> silently yields NoData),
    and are /vsicurl URLs so the GeoEngine worker streams from S3.
    """
    import calendar as _cal

    band = 0
    bands_xml = []
    for file_path in files:
        key = file_path[len(S3_BUCKET) + 1:] if file_path.startswith(S3_BUCKET + "/") else file_path
        source = f"/vsicurl/https://{S3_BUCKET}.s3.us-west-2.amazonaws.com/{key}"
        year = int([t for t in key.split("/")[-1].split("_") if t.isdigit()][-1])
        days = 366 if _cal.isleap(year) else 365
        for d in range(1, days + 1):
            band += 1
            bands_xml.append(
                f'<VRTRasterBand dataType="Float32" band="{band}">'
                f'<NoDataValue>1e20</NoDataValue>'
                f'<ComplexSource>'
                f'<SourceFilename relativeToVRT="0">{source}</SourceFilename>'
                f'<SourceBand>{d}</SourceBand>'
                f'<SourceProperties RasterXSize="1440" RasterYSize="600" DataType="Float32" BlockXSize="1440" BlockYSize="600" />'
                f'<SrcRect xOff="0" yOff="0" xSize="1440" ySize="600" />'
                f'<DstRect xOff="0" yOff="0" xSize="1440" ySize="600" />'
                f'<NODATA>1e20</NODATA>'
                f'</ComplexSource>'
                f'</VRTRasterBand>'
            )

    # NEX-GDDP-CMIP6 real grid: 0.25 deg, 1440x600, lon 0..360, lat -60..90
    vrt = (
        '<VRTDataset rasterXSize="1440" rasterYSize="600">\n'
        '  <SRS>EPSG:4326</SRS>\n'
        '  <GeoTransform>0.0, 0.25, 0, 90.0, 0, -0.25</GeoTransform>\n'
        + "\n".join(bands_xml)
        + "\n</VRTDataset>\n"
    )

    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(vrt)
    print(f"Created VRT: {output_path} ({band} bands)")
    return band


In [6]:

# Create VRTs for all combinations (staging dir as fallback)
assert DATA_DIR is not None, \
    "No volume path - place refs via upload or mount the volume before running"
OUTPUT_DIR = DATA_DIR

for model in MODELS:
    for scenario in SCENARIOS:
        for variable in VARIABLES:
            print(f"\nProcessing: {model['id']} / {scenario} / {variable}")
            files = list_netcdf_files(
                model["id"], scenario, variable, model["variant"], model["grid"],
            )
            if not files:
                print("No files found, skipping")
                continue
            output_path = OUTPUT_DIR / model["id"] / scenario / f"{variable}.vrt"
            create_vrt(files, output_path)



Processing: ACCESS-CM2 / historical / tasmax
Created VRT: /mnt/peter/geo_data/nexgddp-cmip6/ACCESS-CM2/historical/tasmax.vrt (23741 bands)

Processing: ACCESS-CM2 / historical / tasmin
Created VRT: /mnt/peter/geo_data/nexgddp-cmip6/ACCESS-CM2/historical/tasmin.vrt (23741 bands)

Processing: ACCESS-CM2 / historical / pr
Created VRT: /mnt/peter/geo_data/nexgddp-cmip6/ACCESS-CM2/historical/pr.vrt (23741 bands)

Processing: ACCESS-CM2 / ssp245 / tasmax
Created VRT: /mnt/peter/geo_data/nexgddp-cmip6/ACCESS-CM2/ssp245/tasmax.vrt (31411 bands)

Processing: ACCESS-CM2 / ssp245 / tasmin
Created VRT: /mnt/peter/geo_data/nexgddp-cmip6/ACCESS-CM2/ssp245/tasmin.vrt (31411 bands)

Processing: ACCESS-CM2 / ssp245 / pr
Created VRT: /mnt/peter/geo_data/nexgddp-cmip6/ACCESS-CM2/ssp245/pr.vrt (31411 bands)


## Register Datasets in GeoEngine

Register each reference file as a GeoEngine dataset.

In [7]:

def build_netcdf_cf_metadata(variable, scenario, year_start, year_end, reference_path):
    """Build NetCDF-CF metadata for a band-concatenating VRT registration.

    Backing VRT has one band per day since year_start; NetCDF-CF maps time ->
    band via band = 1 + band_offset + steps_between. Requires the VRT's sources
    (https ... .nc via /vsicurl) to pass the worker's curl extension allowlist.
    """
    config = VARIABLE_CONFIGS[variable]

    # NEX-GDDP-CMIP6 actual grid: 0.25 deg, 1440x600, lon 0..360, lat -60..90
    n_lat, n_lon, dx = 600, 1440, 0.25
    x_min, y_max = 0.0, 90.0

    measurement = geoc.Measurement(
        actual_instance=geoc.ContinuousMeasurement(
            type="continuous", measurement=config["unit"], unit=config["unit"]
        )
    )
    start_ms = int(datetime(year_start, 1, 1).timestamp() * 1000)
    end_ms = int(datetime(year_end, 1, 1).timestamp() * 1000)  # non-inclusive
    time_dimension = geoc.TimeDimension(
        actual_instance=geoc.Regular(
            origin=start_ms,
            step=geoc.TimeStep(granularity="days", step=1),
            type="regular",
        )
    )
    geo_transform = geoc.GeoTransform(
        origin_coordinate=geoc.Coordinate2D(x=x_min, y=y_max),
        x_pixel_size=dx,
        y_pixel_size=-dx,
    )
    result_descriptor = geoc.RasterResultDescriptor(
        data_type="F32",
        spatial_reference="EPSG:4326",
        time=geoc.TimeDescriptor(
            bounds=geoc.TimeInterval(start=start_ms, end=end_ms),
            dimension=time_dimension,
        ),
        spatial_grid=geoc.SpatialGridDescriptor(
            spatialGrid=geoc.SpatialGridDefinition(
                geoTransform=geo_transform,
                gridBounds=geoc.GridBoundingBox2D(
                    topLeftIdx=geoc.GridIdx2D(xIdx=0, yIdx=0),
                    bottomRightIdx=geoc.GridIdx2D(xIdx=n_lon - 1, yIdx=n_lat - 1),
                ),
            ),
            descriptor="source",
        ),
        measurement=measurement,
        bands=[geoc.RasterBandDescriptor(name=variable, measurement=measurement)],
    )

    params = geoc.GdalDatasetParameters(
        file_path=str(reference_path),
        rasterband_channel=1,
        geo_transform=geo_transform,
        width=n_lon,
        height=n_lat,
        file_not_found_handling="NoData",
        gdal_config_options=[
            ["CPL_VSIL_CURL_ALLOWED_EXTENSIONS", ".tif,.tiff,.TIF,.jp2,.ovr,.nc"],
        ],
    )

    return geoc.GdalMetadataNetCdfCf(
        type="GdalMetaDataNetCdfCf",
        result_descriptor=result_descriptor,
        params=params,
        start=start_ms,
        end=end_ms,
        step=geoc.TimeStep(granularity="days", step=1),
        band_offset=0,
    )

def register_dataset(model_id, scenario, variable, reference_path, year_start, year_end):
    """Register a dataset in GeoEngine using the volume add-or-replace API."""
    config = VARIABLE_CONFIGS[variable]
    dataset_name = f"nexgddp_cmip6_{model_id}_{scenario}_{variable}"
    display_name = f"NEX-GDDP-CMIP6 {config['name']} ({scenario})"

    properties = ge.AddDatasetProperties(
        name=dataset_name,
        display_name=display_name,
        description=(
            f"Daily NEX-GDDP-CMIP6 {config['name']} projections for {config['measurement']} ({config['unit']}). "
            f"Model: {model_id}, Scenario: {scenario}, grid: VRT over yearly NetCDF files"
        ),
        source_operator="GdalSource",
        symbology=ge.RasterSymbology(
            raster_colorizer=ge.SingleBandRasterColorizer(
                band=0,
                band_colorizer=ge.Colorizer.linear_with_mpl_cmap(
                    color_map=config["color_map"],
                    min_max=(config["min"], config["max"]),
                    n_steps=10,
                ),
            ),
        ),
    )

    meta_data = geoc.MetaDataDefinition(
        build_netcdf_cf_metadata(variable, scenario, year_start, year_end, reference_path)
    )

    ge.add_or_replace_dataset_with_permissions(
        volume,
        properties,
        meta_data,
        permission_tuples=permission_tuples,
        replace_existing=True,
        timeout=600,
    )
    print(f"Registered: {dataset_name}")
    return dataset_name


In [8]:

# Register all datasets
YEAR_RANGES = {"historical": (1950, 2015), "ssp245": (2015, 2101), "ssp585": (2015, 2101)}
registered_datasets = []

for model in MODELS:
    for scenario in SCENARIOS:
        year_start, year_end = YEAR_RANGES[scenario]
        for variable in VARIABLES:
            rel_ref = Path("nexgddp-cmip6") / model["id"] / scenario / f"{variable}.vrt"
            # file is written under DATA_DIR (= volume_root/nexgddp-cmip6);
            # rel_ref is relative to the volume root, hence includes the prefix
            ref_abs = DATA_DIR / model["id"] / scenario / f"{variable}.vrt"
            if ref_abs.exists():
                dataset_name = register_dataset(
                    model["id"], scenario, variable,
                    rel_ref,  # path is relative to the volume root
                    year_start, year_end,
                )
                registered_datasets.append(dataset_name)

print(f"\nTotal datasets: {len(registered_datasets)}")


Registered: nexgddp_cmip6_ACCESS-CM2_historical_tasmax
Registered: nexgddp_cmip6_ACCESS-CM2_historical_tasmin
Registered: nexgddp_cmip6_ACCESS-CM2_historical_pr
Registered: nexgddp_cmip6_ACCESS-CM2_ssp245_tasmax
Registered: nexgddp_cmip6_ACCESS-CM2_ssp245_tasmin
Registered: nexgddp_cmip6_ACCESS-CM2_ssp245_pr

Total datasets: 6


## Verification

Verify that GDAL can read the reference files.

In [9]:

import os
import rasterio

os.environ["CPL_VSIL_CURL_ALLOWED_EXTENSIONS"] = ".tif,.tiff,.TIF,.jp2,.ovr,.nc"

test_vrt = DATA_DIR / MODELS[0]["id"] / SCENARIOS[0] / f"{VARIABLES[0]}.vrt"
if test_vrt.exists():
    try:
        with rasterio.open(test_vrt) as src:
            print(f"Bands: {src.count}, CRS: {src.crs}, Bounds: {src.bounds}")
            data = src.read(1)
            print(f"Band 1 read OK: min={data.min():.1f} max={data.max():.1f} (streams first day from S3)")
    except Exception as e:
        print(f"Failed to open VRT: {e}")
else:
    print(f"Test VRT file not found: {test_vrt}")

# Confirm GeoEngine sees the registered dataset
for ds in registered_datasets[:3]:
    info = ge.dataset_info_by_name(ds)
    print(f"{ds}: {info is not None}")


Bands: 23741, CRS: EPSG:4326, Bounds: BoundingBox(left=0.0, bottom=-60.0, right=360.0, top=90.0)
Band 1 read OK: min=219.0 max=100000002004087734272.0 (streams first day from S3)
nexgddp_cmip6_ACCESS-CM2_historical_tasmax: True
nexgddp_cmip6_ACCESS-CM2_historical_tasmin: True
nexgddp_cmip6_ACCESS-CM2_historical_pr: True


## Next Steps

1. Verify that the GeoEngine worker can read the VRTs (needs GDAL >= 3.8 in the container image)
2. Update the `register_dataset` function with proper metadata extraction
3. Run the full registration (set `DEV_ONLY = False`) to create all 45 datasets, then copy
`nexgddp_models.full.toml` over `nexgddp_models.toml`
4. Test the climate-risk process with the new datasets